# Import

In [ ]:
import pandas as pd
import tensorflow as tf
import numpy as np
from sklearn.preprocessing import OrdinalEncoder , StandardScaler

import matplotlib.pyplot as plt
import seaborn as sns


## Config

In [ ]:
gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Enabled memory growth for {gpu}")
    except Exception as e:
        print(e)

# Functions

In [ ]:
global_split_index_1 = 0
global_split_index_2 = 0

In [ ]:
def make_ds(series, seq_length, batch_size, target, shuffle=False):
    x = series.drop(columns=target).to_numpy()
    y = series[target].to_numpy()    # always slice from the numpy array
    y = y.astype("float32")

    return tf.keras.utils.timeseries_dataset_from_array(
        data=x,
        targets=y,
        sequence_length=seq_length,
        batch_size=batch_size,
        shuffle=shuffle,
        seed=42
    )

def train_valid_test_split(df, train_pct=0.75, valid_pct=0.10):

    split_index_1 = int(len(df) * train_pct)
    split_index_2 = int(len(df) * (train_pct+valid_pct))

    global global_split_index_1
    global global_split_index_2

    global_split_index_1 = split_index_1
    global_split_index_2 = split_index_2

    train_df = df.iloc[:split_index_1].copy()
    valid_df = df.iloc[split_index_1:split_index_2].copy()
    test_df = df.iloc[split_index_2:].copy()


    print(f"""
        split index 1: {split_index_1}
        split index 2: {split_index_2}""")
    return (train_df, valid_df, test_df)

def reuse_split(df):

    global global_split_index_1
    global global_split_index_2

    train_df = df.iloc[:global_split_index_1].copy()
    valid_df = df.iloc[global_split_index_1:global_split_index_2].copy()
    test_df = df.iloc[global_split_index_2:].copy()

    return (train_df, valid_df, test_df)

# Load Data

## Tracks

In [ ]:
Saudi_Arabian_df_fp1 = pd.read_csv("Rounds/weather features/Round_2_Saudi Arabian Grand Prix_FP1.csv")
Saudi_Arabian_df_fp1["Session"] = 1
Saudi_Arabian_df_fp2 = pd.read_csv("Rounds/weather features/Round_2_Saudi Arabian Grand Prix_FP2.csv")
Saudi_Arabian_df_fp2["Session"] = 2
Saudi_Arabian_df_fp3 = pd.read_csv("Rounds/weather features/Round_2_Saudi Arabian Grand Prix_FP3.csv")
Saudi_Arabian_df_fp3["Session"] = 3
Saudi_Arabian_df_race = pd.read_csv("Rounds/weather features/Round_2_Saudi Arabian Grand Prix_Race.csv")
Saudi_Arabian_df_race["Session"] = 4

In [ ]:
Saudi_Arabian_df = pd.concat([Saudi_Arabian_df_fp1,Saudi_Arabian_df_fp2,Saudi_Arabian_df_fp3,Saudi_Arabian_df_race])

# Filters

In [ ]:
# Saudi_Arabian_df = Saudi_Arabian_df[Saudi_Arabian_df["TrackStatus"] == 1]

## Tyre mappings

In [ ]:
Saudi_Arabian_tyre_mapping = {
    "HARD": 2,
    "MEDIUM": 3,
    "SOFT": 4,
}
Saudi_Arabian_df["Compound"] = Saudi_Arabian_df["Compound"].replace(Saudi_Arabian_tyre_mapping)


In [ ]:
Saudi_Arabian_df.to_csv("Saudi_Arabian_complete.csv")

# column management

In [ ]:
Saudi_Arabian_Ver_df = Saudi_Arabian_df.copy()

In [ ]:
Saudi_Arabian_Ver_df = Saudi_Arabian_Ver_df[Saudi_Arabian_Ver_df["Driver"] == "VER"]
Saudi_Arabian_Ver_df = Saudi_Arabian_Ver_df.drop(columns=["Unnamed: 0","DriverNumber","Time","IsPersonalBest","LapStartTime","LapStartDate","Position","Deleted","DeletedReason", "FastF1Generated", "IsAccurate",'Sector1SessionTime',
       'Sector2SessionTime', 'Sector3SessionTime','FreshTyre',"SpeedI1","SpeedI2","SpeedFL","SpeedST","Team","LapTime","PitOutTime","PitInTime","Rainfall"])

In [ ]:
Saudi_Arabian_Ver_df

no more sector time melt, keep them as columns and now they are 3 targets for 3 models. or maybe 3 targets for one?

In [ ]:
Saudi_Arabian_Ver_df = Saudi_Arabian_Ver_df.sort_values(            # sorting by lap then by sector number within lap
    by=["Session","LapNumber"],
    ascending=True
)

In [ ]:
Saudi_Arabian_Ver_df = Saudi_Arabian_Ver_df.reset_index(drop=True)

window_size = 5


for i in range (1,4):
    Saudi_Arabian_Ver_df[f"Sector{i}Time"] = pd.to_timedelta(Saudi_Arabian_Ver_df[f"Sector{i}Time"])
    Saudi_Arabian_Ver_df[f"Sector{i}Time"] = Saudi_Arabian_Ver_df[f"Sector{i}Time"].dt.total_seconds()

    # Saudi_Arabian_Ver_df = Saudi_Arabian_Ver_df.dropna(subset=[f"Sector{i}Time"])                                                 #lazy destructive method
    roll_medians = Saudi_Arabian_Ver_df[f"Sector{i}Time"].rolling(window=window_size, center=True, min_periods=1).median()
    Saudi_Arabian_Ver_df[f"Sector{i}Time"] = Saudi_Arabian_Ver_df[f"Sector{i}Time"].fillna(roll_medians)


# Splitting

In [ ]:
Train_test_valid_sector1_df = Saudi_Arabian_Ver_df.copy()
Train_test_valid_sector2_df = Saudi_Arabian_Ver_df.copy()
Train_test_valid_sector3_df = Saudi_Arabian_Ver_df.copy()

Train_test_valid_sector1_df = Train_test_valid_sector1_df.drop(columns=["Sector2Time","Sector3Time"])
Train_test_valid_sector2_df = Train_test_valid_sector2_df.drop(columns=["Sector1Time","Sector3Time"])
Train_test_valid_sector3_df = Train_test_valid_sector3_df.drop(columns=["Sector1Time","Sector2Time"])

In [ ]:
Ver_train_sector1, Ver_valid_sector1, Ver_test_sector1 = train_valid_test_split(Train_test_valid_sector1_df)
Ver_train_sector2, Ver_valid_sector2, Ver_test_sector2 = reuse_split(Train_test_valid_sector2_df,)
Ver_train_sector3, Ver_valid_sector3, Ver_test_sector3 = reuse_split(Train_test_valid_sector3_df)

# Encoding and numerical formatting

In [ ]:
ints_that_should_not_be_floats = ['LapNumber','Stint','TyreLife']
categorical_cols = ["Compound", "Driver", "TrackStatus"]

ordinal_encoder_1 = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,            # all unseen categories map to -1
    encoded_missing_value=-2     # missing values map to -2
)
ordinal_encoder_2 = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,            
    encoded_missing_value=-2     
)
ordinal_encoder_3 = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1,            
    encoded_missing_value=-2    
)

## S1

ordinal_encoder_1.fit(Ver_train_sector1[categorical_cols])

Ver_train_sector1[categorical_cols] = ordinal_encoder_1.transform(Ver_train_sector1[categorical_cols]).astype("int32")
Ver_train_sector1[ints_that_should_not_be_floats] = Ver_train_sector1[ints_that_should_not_be_floats].astype("int32")

Ver_valid_sector1[categorical_cols] = ordinal_encoder_1.transform(Ver_valid_sector1[categorical_cols]).astype("int32")
Ver_valid_sector1[ints_that_should_not_be_floats] = Ver_valid_sector1[ints_that_should_not_be_floats].astype("int32")

Ver_test_sector1[categorical_cols] = ordinal_encoder_1.transform(Ver_test_sector1[categorical_cols]).astype("int32")
Ver_test_sector1[ints_that_should_not_be_floats] = Ver_test_sector1[ints_that_should_not_be_floats].astype("int32")

## S2

ordinal_encoder_2.fit(Ver_train_sector2[categorical_cols])

Ver_train_sector2[categorical_cols] = ordinal_encoder_2.transform(Ver_train_sector2[categorical_cols]).astype("int32")
Ver_train_sector2[ints_that_should_not_be_floats] = Ver_train_sector2[ints_that_should_not_be_floats].astype("int32")

Ver_valid_sector2[categorical_cols] = ordinal_encoder_2.transform(Ver_valid_sector2[categorical_cols]).astype("int32")
Ver_valid_sector2[ints_that_should_not_be_floats] = Ver_valid_sector2[ints_that_should_not_be_floats].astype("int32")

Ver_test_sector2[categorical_cols] = ordinal_encoder_2.transform(Ver_test_sector2[categorical_cols]).astype("int32")
Ver_test_sector2[ints_that_should_not_be_floats] = Ver_test_sector2[ints_that_should_not_be_floats].astype("int32")

## S3

ordinal_encoder_3.fit(Ver_train_sector3[categorical_cols])

Ver_train_sector3[categorical_cols] = ordinal_encoder_3.transform(Ver_train_sector3[categorical_cols]).astype("int32")
Ver_train_sector3[ints_that_should_not_be_floats] = Ver_train_sector3[ints_that_should_not_be_floats].astype("int32")

Ver_valid_sector3[categorical_cols] = ordinal_encoder_3.transform(Ver_valid_sector3[categorical_cols]).astype("int32")
Ver_valid_sector3[ints_that_should_not_be_floats] = Ver_valid_sector3[ints_that_should_not_be_floats].astype("int32")

Ver_test_sector3[categorical_cols] = ordinal_encoder_3.transform(Ver_test_sector3[categorical_cols]).astype("int32")
Ver_test_sector3[ints_that_should_not_be_floats] = Ver_test_sector3[ints_that_should_not_be_floats].astype("int32")


# Scaling

In [ ]:
to_be_standardised = ['LapNumber','Stint','TyreLife','Session',"AirTemp","Humidity","Pressure","TrackTemp","WindDirection","WindSpeed"]

standard_scaler_1 = StandardScaler()
standard_scaler_2 = StandardScaler()        # not necessary but makes it more diffiult to perform transformations with incorectly fit scalars later 
standard_scaler_3 = StandardScaler()

## S1

standard_scaler_1.fit(Ver_train_sector1[to_be_standardised])

Ver_train_sector1[to_be_standardised] = standard_scaler_1.transform(Ver_train_sector1[to_be_standardised]) 
Ver_valid_sector1[to_be_standardised] = standard_scaler_1.transform(Ver_valid_sector1[to_be_standardised])
Ver_test_sector1[to_be_standardised] = standard_scaler_1.transform(Ver_test_sector1[to_be_standardised]) 

## S2

standard_scaler_2.fit(Ver_train_sector2[to_be_standardised])

Ver_train_sector2[to_be_standardised] = standard_scaler_2.transform(Ver_train_sector2[to_be_standardised]) 
Ver_valid_sector2[to_be_standardised] = standard_scaler_2.transform(Ver_valid_sector2[to_be_standardised])
Ver_test_sector2[to_be_standardised] = standard_scaler_2.transform(Ver_test_sector2[to_be_standardised]) 

## S3

standard_scaler_3.fit(Ver_train_sector3[to_be_standardised])

Ver_train_sector3[to_be_standardised] = standard_scaler_3.transform(Ver_train_sector3[to_be_standardised]) 
Ver_valid_sector3[to_be_standardised] = standard_scaler_3.transform(Ver_valid_sector3[to_be_standardised])
Ver_test_sector3[to_be_standardised] = standard_scaler_3.transform(Ver_test_sector3[to_be_standardised]) 

# datasets

In [ ]:
seq_length = 7
batchsize = 64

Ver_train_sector1_ds = make_ds(Ver_train_sector1, seq_length, batchsize, "Sector1Time", shuffle=True)               # data, seq length, batchsize,shuffle # needs to be numpy format
Ver_valid_sector1_ds = make_ds(Ver_valid_sector1, seq_length, batchsize, "Sector1Time", shuffle=False)
Ver_test_sector1_ds = make_ds(Ver_test_sector1, seq_length, batchsize, "Sector1Time", shuffle=False)

Ver_train_sector2_ds = make_ds(Ver_train_sector2, seq_length, batchsize, "Sector2Time", shuffle=True)               # data, seq length, batchsize,shuffle # needs to be numpy format
Ver_valid_sector2_ds = make_ds(Ver_valid_sector2, seq_length, batchsize, "Sector2Time", shuffle=False)
Ver_test_sector2_ds = make_ds(Ver_test_sector2, seq_length, batchsize, "Sector2Time", shuffle=False)

Ver_train_sector3_ds = make_ds(Ver_train_sector3, seq_length, batchsize, "Sector3Time", shuffle=True)               # data, seq length, batchsize,shuffle # needs to be numpy format
Ver_valid_sector3_ds = make_ds(Ver_valid_sector3, seq_length, batchsize, "Sector3Time", shuffle=False)
Ver_test_sector3_ds = make_ds(Ver_test_sector3, seq_length, batchsize, "Sector3Time", shuffle=False)


# Sanity Checks

In [ ]:
for batch_x, batch_y in Ver_train_sector1_ds.take(1):
    print("X shape S1:", batch_x.shape)                        # (viable windows, seq_length, features)
    print("y shape S1:", batch_y.shape)                        # (batch,) for single target

for batch_x, batch_y in Ver_valid_sector1_ds.take(1):
    print("X shape S2:", batch_x.shape)                        # (batch, seq_length, features)
    print("y shape S2:", batch_y.shape)                        # (batch,) for single target

for batch_x, batch_y in Ver_test_sector1_ds.take(1):
    print("X shape S3:", batch_x.shape)                        # (batch, seq_length, features)
    print("y shape S3:", batch_y.shape)                        # (batch,) for single target

In [ ]:
# Get the first window and its target

for batch_x, batch_y in Ver_train_sector1_ds.take(1):
    first_window = batch_x[0].numpy()
    first_target = batch_y[0].numpy()
    print("First window last row (features):", first_window[-1])
    print("First target (Sector1Time):", first_target)


In [ ]:
print("train windows:", tf.data.experimental.cardinality(Ver_train_sector1_ds))
print("valid windows:", tf.data.experimental.cardinality(Ver_valid_sector1_ds))
print("test  windows:", tf.data.experimental.cardinality(Ver_test_sector1_ds))   


# Modelling

## Architecture

### Sector 1 model

In [ ]:
Ver_sector_1_model = tf.keras.Sequential([
    tf.keras.layers.GRU(64, return_sequences=True,dropout=0.2, recurrent_dropout=0.2,               ## return sequences is needed otherwise only last hidden state is returned        
                        input_shape=(seq_length, len(Ver_train_sector1.columns)-1)),
    tf.keras.layers.LayerNormalization(),
    
    tf.keras.layers.GRU(64, dropout=0.2, recurrent_dropout=0.2),
    tf.keras.layers.LayerNormalization(),
    
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    
    tf.keras.layers.Dense(1)
])

### Sector 2 model

In [ ]:
Ver_sector_2_model = tf.keras.Sequential([
    tf.keras.layers.GRU(64, return_sequences=True,dropout=0.2, recurrent_dropout=0.2,                   ## return sequences is needed otherwise only last hidden state is returned
                        input_shape=(seq_length, len(Ver_train_sector1.columns)-1)),
    tf.keras.layers.LayerNormalization(),
    
    tf.keras.layers.GRU(64, dropout=0.2, recurrent_dropout=0.2),
    tf.keras.layers.LayerNormalization(),
    
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    
    tf.keras.layers.Dense(1)                                                                            # no activation function by default # the dense layer is crtitical other wise the rnn with small datasets will have its outputs reduced to 1 or -1, blame tanh
])

### Sector 3 model

In [ ]:
Ver_sector_3_model = tf.keras.Sequential([
     tf.keras.layers.GRU(64, return_sequences=True,dropout=0.2, recurrent_dropout=0.2,                   ## return sequences is needed otherwise only last hidden state is returned
                        input_shape=(seq_length, len(Ver_train_sector1.columns)-1)),
    tf.keras.layers.LayerNormalization(),
    
    tf.keras.layers.GRU(64, dropout=0.2, recurrent_dropout=0.2),
    tf.keras.layers.LayerNormalization(),
    
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.2),
    
    tf.keras.layers.Dense(1)                                                                            # no activation function by default often used for regression work. # the dense layer is crtitical other wise the rnn with small datasets will have its outputs reduced to 1 or -1, blame tanh
])

# gradient clipping?????

## Running Models

### Sector 1

In [ ]:
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_mae",
    patience=50,
    restore_best_weights=True)

opt = tf.keras.optimizers.Adam(
    learning_rate=0.001,
    clipvalue = 1.0)

Ver_sector_1_model.compile(
    loss=tf.keras.losses.Huber(),
    optimizer=opt,
    metrics=["mae"])

history = Ver_sector_1_model.fit(
    Ver_train_sector1_ds,
    validation_data = Ver_valid_sector1_ds,
    callbacks = early_stopping_cb,
    epochs=250)

### Sector 2

In [ ]:
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_mae",
    patience=50,
    restore_best_weights=True)

opt = tf.keras.optimizers.Adam(
    learning_rate=0.001,
    clipvalue = 1.0)

Ver_sector_2_model.compile(
    loss=tf.keras.losses.Huber(),
    optimizer=opt,
    metrics=["mae"])

history = Ver_sector_2_model.fit(
    Ver_train_sector2_ds,
    validation_data = Ver_valid_sector2_ds,
    callbacks = early_stopping_cb,
    epochs=250)

### Sector 3

In [ ]:
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor="val_mae",
    patience=50,
    restore_best_weights=True)

opt = tf.keras.optimizers.Adam(
    learning_rate=0.001,
    clipvalue = 1.0)

Ver_sector_3_model.compile(
    loss=tf.keras.losses.Huber(),
    optimizer=opt,
    metrics=["mae"])

history = Ver_sector_3_model.fit(
    Ver_train_sector3_ds,
    validation_data = Ver_valid_sector3_ds,
    callbacks = early_stopping_cb,
    epochs=250)

## Results

In [ ]:
test_loss_1, test_mae_1 = Ver_sector_1_model.evaluate(Ver_test_sector1_ds)
print("Sector 1 Test MAE:", test_mae_1)


In [ ]:
test_loss_2, test_mae_2 = Ver_sector_2_model.evaluate(Ver_test_sector2_ds)
print("Sector 2 Test MAE:", test_mae_2)


In [ ]:
test_loss_3, test_mae_3 = Ver_sector_3_model.evaluate(Ver_test_sector3_ds)
print("Sector 3 Test MAE:", test_mae_3)


In [ ]:
preds_1 = Ver_sector_1_model.predict(Ver_test_sector1_ds)
print(f"Sector 1 preds:{preds_1}")

In [ ]:
preds_2 = Ver_sector_2_model.predict(Ver_test_sector2_ds)
print(f"Sector 2 preds:{preds_2}")

In [ ]:
preds_3 = Ver_sector_3_model.predict(Ver_test_sector3_ds)
print(f"Sector 3 preds:{preds_3}")

# Prediction Inspection

testing split starts at split_index_2 and will start predicting from row index (index_2 + sequence length)

In [ ]:
Test_pred_real = Saudi_Arabian_Ver_df.iloc[global_split_index_2+seq_length-1:][["LapNumber","Sector1Time","Sector2Time","Sector3Time","TrackStatus","Compound"]]       # sequence length starts at 0
Test_pred_real[["Sector1_pred","Sector2_pred","Sector3_pred"]] = np.hstack([preds_1,preds_2,preds_3])
Test_pred_real["LapTime"] = Test_pred_real["Sector1Time"] + Test_pred_real["Sector2Time"] + Test_pred_real["Sector3Time"]
Test_pred_real["Predicted_LapTime"] = Test_pred_real["Sector1_pred"] + Test_pred_real["Sector2_pred"] + Test_pred_real["Sector3_pred"]

Test_pred_real = Test_pred_real[["LapNumber","LapTime","Predicted_LapTime","Sector1Time","Sector1_pred","Sector2Time","Sector2_pred","Sector3Time","Sector3_pred","Compound","TrackStatus"]]

In [ ]:
Test_pred_real

# Plots

In [ ]:


plt.figure(figsize=(16, 12))


# 1. LapTime vs Predicted LapTime

plt.subplot(2, 2, 1)
sns.lineplot(data=Test_pred_real, x="LapNumber", y="LapTime", label="Actual LapTime")
sns.lineplot(data=Test_pred_real, x="LapNumber", y="Predicted_LapTime", label="Predicted LapTime")
plt.title("Lap Time vs Predicted Lap Time")
plt.xlabel("Lap Number")
plt.ylabel("Time (s)")


# 2. Sector 1

plt.subplot(2, 2, 2)
sns.lineplot(data=Test_pred_real, x="LapNumber", y="Sector1Time", label="Actual S1")
sns.lineplot(data=Test_pred_real, x="LapNumber", y="Sector1_pred", label="Predicted S1")
plt.title("Sector 1 vs Predicted Sector 1")
plt.xlabel("Lap Number")
plt.ylabel("Time (s)")


# 3. Sector 2

plt.subplot(2, 2, 3)
sns.lineplot(data=Test_pred_real, x="LapNumber", y="Sector2Time", label="Actual S2")
sns.lineplot(data=Test_pred_real, x="LapNumber", y="Sector2_pred", label="Predicted S2")
plt.title("Sector 2 vs Predicted Sector 2")
plt.xlabel("Lap Number")
plt.ylabel("Time (s)")


# 4. Sector 3

plt.subplot(2, 2, 4)
sns.lineplot(data=Test_pred_real, x="LapNumber", y="Sector3Time", label="Actual S3")
sns.lineplot(data=Test_pred_real, x="LapNumber", y="Sector3_pred", label="Predicted S3")
plt.title("Sector 3 vs Predicted Sector 3")
plt.xlabel("Lap Number")
plt.ylabel("Time (s)")

plt.tight_layout()
plt.show()

# functions

In [ ]:
def create_sector_prediction_frames(df):
    sector1_df = df.copy().drop(columns=["Sector2Time", "Sector3Time"])
    sector2_df = df.copy().drop(columns=["Sector1Time", "Sector3Time"])
    sector3_df = df.copy().drop(columns=["Sector1Time", "Sector2Time"])
    return sector1_df, sector2_df, sector3_df

def prepare_prediction_df(df, encoder, scaler,
                          categorical_cols, ints_that_should_not_be_floats,
                          to_be_standardised):
    
    # Encode categorical features
    df[categorical_cols] = encoder.transform(df[categorical_cols]).astype("int32")

    # Ensure clean integer columns
    df[ints_that_should_not_be_floats] = df[ints_that_should_not_be_floats].astype("int32")

    # Apply the correct standard scaler
    df[to_be_standardised] = scaler.transform(df[to_be_standardised])

    return df


def prepare_prediction_datasets(new_df):
    
    #  Sector-specific copies

    pred_s1, pred_s2, pred_s3 = create_sector_prediction_frames(new_df)

    #  Apply transformation pipeline (no fitting)

    pred_s1 = prepare_prediction_df(
        pred_s1,
        encoder=ordinal_encoder_1,
        scaler=standard_scaler_1,
        categorical_cols=categorical_cols,
        ints_that_should_not_be_floats=ints_that_should_not_be_floats,
        to_be_standardised=to_be_standardised
    )

    pred_s2 = prepare_prediction_df(
        pred_s2,
        encoder=ordinal_encoder_2,
        scaler=standard_scaler_2,
        categorical_cols=categorical_cols,
        ints_that_should_not_be_floats=ints_that_should_not_be_floats,
        to_be_standardised=to_be_standardised
    )

    pred_s3 = prepare_prediction_df(
        pred_s3,
        encoder=ordinal_encoder_3,
        scaler=standard_scaler_3,
        categorical_cols=categorical_cols,
        ints_that_should_not_be_floats=ints_that_should_not_be_floats,
        to_be_standardised=to_be_standardised
    )

    # --- Convert to prediction datasets (sequence-based) ---
    pred_s1_ds = make_ds(pred_s1, seq_length, batchsize, "Sector1Time", shuffle=False)
    pred_s2_ds = make_ds(pred_s2, seq_length, batchsize, "Sector2Time", shuffle=False)
    pred_s3_ds = make_ds(pred_s3, seq_length, batchsize, "Sector3Time", shuffle=False)

    return pred_s1_ds, pred_s2_ds, pred_s3_ds, pred_s1, pred_s2, pred_s3,



# predicting over race , to supply context to test split

In [ ]:
ver_pred_s1_ds, ver_pred_s2_ds, ver_pred_s3_ds, ver_pred_s1, ver_pred_s2, ver_pred_s3, = prepare_prediction_datasets(Saudi_Arabian_Ver_df[Saudi_Arabian_Ver_df["Session"]==4])

In [ ]:
P_S1 = Ver_sector_1_model.predict(ver_pred_s1_ds)


In [ ]:
P_S2 = Ver_sector_2_model.predict(ver_pred_s2_ds)


In [ ]:
P_S3 = Ver_sector_3_model.predict(ver_pred_s3_ds)


In [ ]:
Race_pred_real = Saudi_Arabian_Ver_df[Saudi_Arabian_Ver_df["Session"] ==4]
Race_pred_real = Race_pred_real.iloc[seq_length-1:][["LapNumber","Sector1Time","Sector2Time","Sector3Time","TrackStatus","Compound"]]       # sequence length starts at 0
Race_pred_real[["Sector1_pred","Sector2_pred","Sector3_pred"]] = np.hstack([P_S1,P_S2,P_S3])
Race_pred_real["LapTime"] = Race_pred_real["Sector1Time"] + Race_pred_real["Sector2Time"] + Race_pred_real["Sector3Time"]
Race_pred_real["Predicted_LapTime"] = Race_pred_real["Sector1_pred"] + Race_pred_real["Sector2_pred"] + Race_pred_real["Sector3_pred"]

Race_pred_real = Race_pred_real[["LapNumber","LapTime","Predicted_LapTime","Sector1Time","Sector1_pred","Sector2Time","Sector2_pred","Sector3Time","Sector3_pred","Compound","TrackStatus"]]

Race_pred_real

# Plotting over race - just for inspection

In [ ]:
plt.figure(figsize=(16, 12))

# 1. LapTime vs Predicted LapTime

plt.subplot(2, 2, 1)
sns.lineplot(data=Race_pred_real, x="LapNumber", y="LapTime", label="Actual LapTime")
sns.lineplot(data=Race_pred_real, x="LapNumber", y="Predicted_LapTime", label="Predicted LapTime")
plt.title("Lap Time vs Predicted Lap Time")
plt.xlabel("Lap Number")
plt.ylabel("Time (s)")

# 2. Sector 1

plt.subplot(2, 2, 2)
sns.lineplot(data=Race_pred_real, x="LapNumber", y="Sector1Time", label="Actual S1")
sns.lineplot(data=Race_pred_real, x="LapNumber", y="Sector1_pred", label="Predicted S1")
plt.title("Sector 1 vs Predicted Sector 1")
plt.xlabel("Lap Number")
plt.ylabel("Time (s)")


# 3. Sector 2

plt.subplot(2, 2, 3)
sns.lineplot(data=Race_pred_real, x="LapNumber", y="Sector2Time", label="Actual S2")
sns.lineplot(data=Race_pred_real, x="LapNumber", y="Sector2_pred", label="Predicted S2")
plt.title("Sector 2 vs Predicted Sector 2")
plt.xlabel("Lap Number")
plt.ylabel("Time (s)")


# 4. Sector 3

plt.subplot(2, 2, 4)
sns.lineplot(data=Race_pred_real, x="LapNumber", y="Sector3Time", label="Actual S3")
sns.lineplot(data=Race_pred_real, x="LapNumber", y="Sector3_pred", label="Predicted S3")
plt.title("Sector 3 vs Predicted Sector 3")
plt.xlabel("Lap Number")
plt.ylabel("Time (s)")

plt.tight_layout()
plt.show()